# Gelatina - Verificacion Formal (interactiva)

Capsula 4. En el Flan probamos **algunos** casos. Aqui la herramienta
**demuestra** la propiedad de *una sola luz* para **todos** los casos (hasta
una profundidad), y cuando metemos el bug, nos entrega el **contraejemplo**:
la secuencia exacta que rompe el diseno.

> Usamos **SymbiYosys** (open source). Corre las celdas en orden.

## 1 - Preparar (descargar la suite de formal)

Bajamos **oss-cad-suite** (trae yosys + sby + los solvers). Pesa unos cientos
de MB, tarda ~1-2 min la primera vez.

In [ ]:
import os, subprocess, requests

# 1) el repo (para el RTL)
if not os.path.isdir('galletas-y-postres-chips'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/Misterio12947/galletas-y-postres-chips.git'], check=True)
RTL = 'galletas-y-postres-chips/common/rtl/semaphore/rtl/semaphore.v'

# 2) oss-cad-suite (yosys + sby + solvers)
if not os.path.isdir('/content/oss-cad-suite'):
    rel = requests.get('https://api.github.com/repos/YosysHQ/oss-cad-suite-build/releases/latest').json()
    url = next(a['browser_download_url'] for a in rel['assets']
               if 'linux-x64' in a['name'] and a['name'].endswith('.tgz'))
    print('Descargando', url.split('/')[-1], '...')
    subprocess.run(['wget','-q',url,'-O','oss.tgz'], check=True)
    subprocess.run(['tar','xf','oss.tgz'], check=True)
    os.remove('oss.tgz')
os.environ['PATH'] = '/content/oss-cad-suite/bin:' + os.environ['PATH']

v = subprocess.run(['sby','--version'], capture_output=True, text=True)
print('sby listo:', (v.stdout + v.stderr).strip())

## 2 - La propiedad (y el reset del analisis)

No damos estimulos: damos una **regla que siempre debe cumplirse**. La misma
de Flan, pero como `assert`.

Ademas incluimos una **secuencia de reset** en el bloque formal. Sin ella, el
motor arrancaria desde un estado *cualquiera* (incluido un estado invalido del
registro de 2 bits) y encontraria un contraejemplo trivial con todas las luces
apagadas. Forzando el reset al inicio, el analisis parte de un estado valido.

Todo va dentro de `` `ifdef FORMAL `` para no estorbar en simulacion ni sintesis:

In [ ]:
ASSERT = (
    '`ifdef FORMAL\n'
    '  // arranque formal: reset activo en el primer ciclo, luego corre normal\n'
    '  reg f_started = 0;\n'
    '  always @(posedge clk) f_started <= 1;\n'
    '  always @(posedge clk) begin\n'
    '    if (!f_started) assume (!rst_n);\n'
    '    else            assume ( rst_n);\n'
    '  end\n'
    '  // propiedad de seguridad: una sola luz, siempre\n'
    '  always @(posedge clk) if (f_started && rst_n) assert (red + yellow + green == 1);\n'
    '`endif\n'
)

def inyectar(src):
    return src.replace('endmodule', ASSERT + 'endmodule', 1)

# RTL correcto + la propiedad
open('semaphore_formal.v', 'w').write(inyectar(open(RTL).read()))
print(ASSERT)

## 3 - Demostrar el RTL correcto

`sby` explora **todos** los casos alcanzables desde el reset hasta la
profundidad dada. Si no encuentra forma de violar la propiedad: **PASS**.

In [ ]:
sby_ok = '''[options]
mode bmc
depth 20

[engines]
smtbmc

[script]
read -formal semaphore_formal.v
prep -top semaforo

[files]
semaphore_formal.v
'''
open('ok.sby','w').write(sby_ok)
r = subprocess.run(['sby','-f','ok.sby'], capture_output=True, text=True)
print('\n'.join(r.stdout.strip().splitlines()[-6:]))

## 4 - Mete el bug -> contraejemplo

Usamos el semaforo **con bug** (en amarillo enciende tambien el verde). Ahora
que el analisis parte de un reset valido, el formal encuentra el fallo **real**
y nos da la **traza exacta**: veras verde y amarillo encendidos a la vez.

La traza la leemos con `vcdvcd` (lee cualquier VCD y localiza las senales por nombre).

In [ ]:
import matplotlib.pyplot as plt

bug = '''module semaforo #(parameter integer T_GREEN=5, T_YELLOW=2, T_RED=5, CW=8)(
  input wire clk, input wire rst_n, output reg red, output reg yellow, output reg green);
  localparam [1:0] S_GREEN=0, S_YELLOW=1, S_RED=2;
  reg [1:0] state; reg [CW-1:0] cnt, limit;
  always @(*) case(state) S_GREEN:limit=T_GREEN; S_YELLOW:limit=T_YELLOW; default:limit=T_RED; endcase
  always @(posedge clk or negedge rst_n)
    if(!rst_n) begin state<=S_GREEN; cnt<=0; end
    else if(cnt==limit-1) begin cnt<=0;
      state<=(state==S_GREEN)?S_YELLOW:(state==S_YELLOW)?S_RED:S_GREEN; end
    else cnt<=cnt+1;
  always @(*) begin red=0; yellow=0; green=0;
    case(state) S_GREEN:green=1; S_YELLOW:begin yellow=1; green=1; end S_RED:red=1; endcase
  end
endmodule
'''
open('semaphore_bug.v','w').write(inyectar(bug))
open('bug.sby','w').write(sby_ok.replace('semaphore_formal.v','semaphore_bug.v'))
r = subprocess.run(['sby','-f','bug.sby'], capture_output=True, text=True)
print('\n'.join(r.stdout.strip().splitlines()[-6:]))

# --- graficar el contraejemplo con vcdvcd (robusto) ---
subprocess.run(['pip','-q','install','vcdvcd'], check=True)
from vcdvcd import VCDVCD
import glob
traza = glob.glob('bug/engine_0/*.vcd')
if not traza:
    print('(no se encontro traza; revisa la salida de sby arriba)')
else:
    vcd = VCDVCD(traza[0])
    # localizar green/yellow/red por nombre, ignorando jerarquia y ancho
    objetivo = {}
    for sig in vcd.signals:
        base = sig.split('.')[-1].split('[')[0].strip()
        if base in ('green','yellow','red') and base not in objetivo:
            objetivo[base] = sig
    print('senales encontradas:', objetivo or 'NINGUNA')
    if not objetivo:
        print('Nombres disponibles (muestra):', vcd.signals[:25])
    else:
        series, tmax = {}, 1
        for nm in ('green', 'yellow', 'red'):
            tv = vcd[objetivo[nm]].tv if nm in objetivo else []
            pts = [(int(t), int(val)) for t, val in tv if val in ('0', '1')]
            series[nm] = pts
            if pts: tmax = max(tmax, pts[-1][0])
        tmax += 1
        print('puntos por senal:', {k: len(v) for k, v in series.items()})
        fig, axes = plt.subplots(3, 1, figsize=(9, 2.6), sharex=True)
        for ax, nm, c in zip(axes, ('green','yellow','red'), ('tab:green','gold','tab:red')):
            xs, ys = [0], [0]
            for tt, vv in series[nm]: xs.append(tt); ys.append(vv)
            xs.append(tmax); ys.append(ys[-1])
            ax.step(xs, ys, where='post', color=c, lw=2)
            ax.set_ylim(-0.2, 1.2); ax.set_yticks([0, 1])
            ax.set_ylabel(nm, rotation=0, ha='right', va='center')
        axes[0].set_title('Contraejemplo del bug: se viola "una sola luz"')
        axes[-1].set_xlabel('tiempo del contraejemplo')
        plt.tight_layout(); plt.show()

## 5 - Lo que sigue

- **PASS** = la propiedad se cumple para todos los casos explorados (no solo los que se nos ocurrieron)
- El **contraejemplo** no solo dice *que* falla: dice *como* llegar al fallo
- Un primo del formal, el **equivalence checking**, volvera tras la sintesis (RTL vs netlist)

🟦 Gelatina &nbsp;->&nbsp; 🟫 Mazapan (Prototipado en FPGA)